In [1]:
from onsager.crystal import Crystal
from onsager.crystalStars import zeroclean
from onsager.OnsagerCalc import *
from onsager.crystal import DB_disp, DB_disp4, pureDBContainer, mixedDBContainer
from onsager.DB_structs import dumbbell, SdPair, jump, connector

In [2]:
import matplotlib.pyplot as plt
import scipy.optimize as sopt
from tqdm import tqdm

In [3]:
# make a BCC lattice
# We'll modify the jumpnetwork to keep only the 60 degree reorientational jumps.
a0 = 1.0
latt = np.array([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]]) * a0
Fe = crystal.Crystal(latt, [[np.array([0., 0., 0.]), np.array([0.5, 0.5, 0.5])]], ["Fe"])
# Now give it the orientations - for BCC it's [110]
o = np.array([1.,1.,0.])/np.linalg.norm(np.array([1.,1.,0.]))*a0/4
famp0 = [o.copy()]
family = [famp0]
pdbcontainer_fe = pureDBContainer(Fe, 0, family)
mdbcontainer_fe = mixedDBContainer(Fe, 0, family)
jcut = np.sqrt(3)*1.01*a0/2.
jset0, jset2 = pdbcontainer_fe.jumpnetwork(jcut, 0.01, 0.01), mdbcontainer_fe.jumpnetwork(jcut, 0.01, 0.01)
print(Fe)

#Lattice:
  a1 = [0.5 0.5 0.5]
  a2 = [-0.5  0.5 -0.5]
  a3 = [-0.5 -0.5  0.5]
#Basis:
  (Fe) 0.0 = [0. 0. 0.]


In [4]:
# Modify jnet0
jnet0 = jset0[0]
jnet0_indexed = jset0[1]
# Let's try to sort the jumps according to closest distance
# except rotational jumps, we don't want them.
z = np.zeros(3)
indices = []

for jt, jlist in enumerate(jnet0):
    if np.allclose(jnet0_indexed[jt][0][1], z):
        continue
    indices.append(jt)
    
def sortkey(entry):
    jmp = jnet0[entry][0]
    or1 = pdbcontainer_fe.iorlist[jmp.state1.iorind][1]
    or2 = pdbcontainer_fe.iorlist[jmp.state2.iorind][1]
    dx = DB_disp(pdbcontainer_fe, jmp.state1, jmp.state2)
    dx1 = np.linalg.norm(jmp.c1*or1/2.)
    dx2 = np.linalg.norm(dx + jmp.c2*or2/2. - jmp.c1*or1/2.)
    dx3 = np.linalg.norm(-jmp.c2*or2/2.)
    return dx1+dx2+dx3
ind_sort = sorted(indices, key=sortkey)

In [5]:
# Let's check if we got the correct jump
print(jnet0[ind_sort[0]][0])

Jump object:
Initial state:
	dumbbell : (i, or) index = 3, lattice vector = [0 0 0]
Final state:
	dumbbell : (i, or) index = 1, lattice vector = [ 0 -1  0]
Jumping from c1 = 1 to c2 = -1



In [6]:
pdbcontainer_fe.iorlist

[(0, array([0.1767767, 0.1767767, 0.       ])),
 (0, array([ 0.1767767, -0.1767767,  0.       ])),
 (0, array([ 0.       , -0.1767767, -0.1767767])),
 (0, array([ 0.       , -0.1767767,  0.1767767])),
 (0, array([0.1767767, 0.       , 0.1767767])),
 (0, array([-0.1767767,  0.       ,  0.1767767]))]

In [7]:
# take only the lowest displacement jump
# that is the jump we want.
jset0new = ([jnet0[ind_sort[0]]], [jnet0_indexed[ind_sort[0]]])

In [8]:
# Now, we modify the mixed dumbbell jumpnetwork to also give the lowest displacement jump
# Modify jnet0
jnet2 = jset2[0]
jnet2_indexed = jset2[1]
# Let's try to sort the jumps according to closest distance
# we don't want the rotational jumps as before.
z = np.zeros(3)
indices2 = []
for jt, jlist in enumerate(jnet2):
    if np.allclose(jnet2_indexed[jt][0][1], z):
        continue
    indices2.append(jt)    
print(indices2)

def sortkey2(entry):
    jmp = jnet2[entry][0]
    or1 = mdbcontainer_fe.iorlist[jmp.state1.db.iorind][1]
    or2 = mdbcontainer_fe.iorlist[jmp.state2.db.iorind][1]
    dx = DB_disp(mdbcontainer_fe, jmp.state1, jmp.state2)
    # c1 and c2 are always +1 for mixed dumbbell jumps.
    dx1 = np.linalg.norm(jmp.c1*or1/2.)
    dx2 = np.linalg.norm(dx + jmp.c2*or2/2. - jmp.c1*or1/2.)
    dx3 = np.linalg.norm(-jmp.c2*or2/2.)
    return dx1+dx2+dx3
ind_sort2 = sorted(indices2, key=sortkey2)
print(ind_sort2)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
[14, 13, 10, 4, 7, 0, 11, 5, 12, 3, 6, 9, 8, 1, 2, 16, 15]


In [9]:
# check if we have the correct type of jump
print(jnet2[ind_sort2[0]][0])

Jump object:
Initial state:
	Solute loctation:basis index = 0, lattice vector = [0 0 0]
	dumbbell : (i, or) index = 1, lattice vector = [0 0 0]
Final state:
	Solute loctation :basis index = 0, lattice vector = [-1 -1 -1]
	dumbbell : (i, or) index = 11, lattice vector = [-1 -1 -1]
Jumping from c1 = 1 to c2 = 1


In [10]:
for tup in mdbcontainer_fe.iorlist:
    print(tup)

(0, array([0.1767767, 0.1767767, 0.       ]))
(0, array([ 0.1767767, -0.1767767,  0.       ]))
(0, array([ 0.       , -0.1767767, -0.1767767]))
(0, array([ 0.       , -0.1767767,  0.1767767]))
(0, array([-0.1767767,  0.1767767,  0.       ]))
(0, array([0.1767767, 0.       , 0.1767767]))
(0, array([-0.1767767,  0.       , -0.1767767]))
(0, array([-0.1767767,  0.       ,  0.1767767]))
(0, array([ 0.1767767,  0.       , -0.1767767]))
(0, array([ 0.       ,  0.1767767, -0.1767767]))
(0, array([-0.1767767, -0.1767767,  0.       ]))
(0, array([0.       , 0.1767767, 0.1767767]))


In [11]:
# take only the lowest displacement jump
# that is the jump we want.
jset2new = ([jnet2[ind_sort2[0]]], [jnet2_indexed[ind_sort2[0]]])

In [12]:
# Now, we construct the Onsager calculator with these non-local jump sets.
start = time.time()
onsagercalculator = dumbbellMediated(pdbcontainer_fe, mdbcontainer_fe, jset0new, jset2new, jcut,
                                     0.01, 0.01, 0.01, NGFmax=4, Nthermo=1)
print("onsager calculator initiation time = {}".format(time.time() - start))

initializing thermo
initializing kin
generating thermodynamic shell
built shell 1: time - 0.026027441024780273
grouped states by symmetry: 0.13930964469909668
built mixed dumbbell stars: 0.0008301734924316406
built jtags2: 0.0006043910980224609
built mixed indexed star: 0.008567094802856445
building star2symlist : 7.939338684082031e-05
building bare, mixed index dicts : 0.0002105236053466797
thermodynamic shell generated: 0.27118921279907227
Total number of states in Thermodynamic Shell - 54, 12
generating kinetic shell
built shell 1: time - 0.0271914005279541
built shell 2: time - 0.7791049480438232
grouped states by symmetry: 1.5233237743377686
built mixed dumbbell stars: 0.0008151531219482422
built jtags2: 0.0003120899200439453
built mixed indexed star: 0.009180545806884766
building star2symlist : 0.0001361370086669922
building bare, mixed index dicts : 0.00027370452880859375
Kinetic shell generated: 3.7187933921813965
Total number of states in Kinetic Shell - 210, 12
generating kin

In [13]:
onsagercalculator.om1types

[0, 0, 0, 0, 0, 0, 0, 0]

In [14]:
# Next, we must also modify the omega3 and omega4 jump lists
jnet43 = onsagercalculator.jnet43
jnet43_indexed = onsagercalculator.jnet43_indexed
# Let's try to sort the jumps according to closest distance
# we don't want the rotational jumps as before.

z = np.zeros(3)
indices43 = []
for jt, jlist in enumerate(jnet43):
    if np.allclose(jnet43_indexed[jt][0][1], z):
        continue
    indices43.append(jt)    
# print(indices43)

def sortkey43(entry):
    jmp = jnet43[entry][0] # This is an omega4 jump
    if not jmp.c2 == -1:
        print(c2)
    or1 = pdbcontainer_fe.iorlist[jmp.state1.db.iorind][1]
    or2 = mdbcontainer_fe.iorlist[jmp.state2.db.iorind][1]
    dx = DB_disp4(pdbcontainer_fe, mdbcontainer_fe, jmp.state1, jmp.state2)
    # remember that c2 is -1 for an omega4 jump
    dx1 = np.linalg.norm(jmp.c1*or1/2.)
    dx2 = np.linalg.norm(dx - or2/2. - jmp.c1*or1/2.)
    dx3 = np.linalg.norm(jmp.c2*or2/2.)
    return dx1+dx2+dx3

ind_sort43 = sorted(indices43, key=sortkey43)
print(ind_sort43)

[16, 24, 9, 22, 7, 14, 8, 18, 0, 12, 25, 4, 11, 17, 20, 3, 10, 1, 6, 19, 2, 15, 5, 23, 21, 13]


In [15]:
# check if we have the correct jump
print(jnet43[ind_sort43[0]][0])

Jump object:
Initial state:
	Solute loctation:basis index = 0, lattice vector = [0 0 0]
	dumbbell : (i, or) index = 0, lattice vector = [ 0  0 -1]
Final state:
	Solute loctation :basis index = 0, lattice vector = [0 0 0]
	dumbbell : (i, or) index = 3, lattice vector = [0 0 0]
Jumping from c1 = -1 to c2 = -1


In [16]:
pdbcontainer_fe.iorlist

[(0, array([0.1767767, 0.1767767, 0.       ])),
 (0, array([ 0.1767767, -0.1767767,  0.       ])),
 (0, array([ 0.       , -0.1767767, -0.1767767])),
 (0, array([ 0.       , -0.1767767,  0.1767767])),
 (0, array([0.1767767, 0.       , 0.1767767])),
 (0, array([-0.1767767,  0.       ,  0.1767767]))]

In [17]:
mdbcontainer_fe.iorlist

[(0, array([0.1767767, 0.1767767, 0.       ])),
 (0, array([ 0.1767767, -0.1767767,  0.       ])),
 (0, array([ 0.       , -0.1767767, -0.1767767])),
 (0, array([ 0.       , -0.1767767,  0.1767767])),
 (0, array([-0.1767767,  0.1767767,  0.       ])),
 (0, array([0.1767767, 0.       , 0.1767767])),
 (0, array([-0.1767767,  0.       , -0.1767767])),
 (0, array([-0.1767767,  0.       ,  0.1767767])),
 (0, array([ 0.1767767,  0.       , -0.1767767])),
 (0, array([ 0.       ,  0.1767767, -0.1767767])),
 (0, array([-0.1767767, -0.1767767,  0.       ])),
 (0, array([0.       , 0.1767767, 0.1767767]))]

In [18]:
onsagercalculator.regenerate43([ind_sort43[0]])

In [19]:
# 1.  First get the rates and thermodynamic data
# All the energies of the "mixed" and pure dumbbells will be the same,
# All the jump rates will be the same
    # Since we have only one type each of omega0, omega2 and omega43 jumps, set their rates to zero.
    # All omega1 rates will be the same as the above rate.
# The "solute" energies will be zero since we are dealing with a chemically identical tracer.
# All interaction energies will be zero.

# 1a. Energies and pre-factors
kT = 1

predb0, enedb0 = np.ones(len(onsagercalculator.vkinetic.starset.pdbcontainer.symorlist)), \
                 np.zeros(len(onsagercalculator.vkinetic.starset.pdbcontainer.symorlist))

preS, eneS = np.ones(
    len(onsagercalculator.vkinetic.starset.crys.sitelist(onsagercalculator.vkinetic.starset.chem))), \
             np.zeros(len(onsagercalculator.vkinetic.starset.crys.sitelist(
                 onsagercalculator.vkinetic.starset.chem)))

# These are the interaction or the excess energies and pre-factors for solutes and dumbbells.
# The energies will all be zero.
preSdb, eneSdb = np.ones(onsagercalculator.thermo.mixedstartindex), \
                 np.zeros(onsagercalculator.thermo.mixedstartindex)

predb2, enedb2 = predb0.copy(), enedb0.copy()

preT0, eneT0 = np.ones(len(onsagercalculator.vkinetic.starset.jnet0)), np.zeros(len(onsagercalculator.jnet0))
preT2, eneT2 = preT0.copy(), eneT0.copy()
preT1, eneT1 = np.ones(len(onsagercalculator.jnet1)), np.array([eneT0[onsagercalculator.om1types[jt]] for jt in
                                                                range(len(onsagercalculator.jnet1))])

preT43, eneT43 = np.ones(len(onsagercalculator.jnet43)), eneT0.copy()

In [20]:
# 1b. Now get the beta*free energy values.
bFdb0, bFdb2, bFS, bFSdb, bFT0, bFT1, bFT2, bFT3, bFT4 = \
    onsagercalculator.preene2betafree(kT, predb0, enedb0, preS, eneS, preSdb, eneSdb, predb2, enedb2,
                                           preT0, eneT0, preT2, eneT2, preT1, eneT1, preT43, eneT43)

In [21]:
len(onsagercalculator.jnet1)

8

In [22]:
# get the probabilities and other data from L_ij
L0bb,(L_uc_aa,L_c_aa), (L_uc_bb,L_c_bb), (L_uc_ab,L_c_ab)=\
onsagercalculator.L_ij(bFdb0, bFT0, bFdb2, bFT2, bFS, bFSdb, bFT1, bFT3, bFT4)

In [23]:
L_aa = L_uc_aa + L_c_aa

In [24]:
L_ab = L_uc_ab + L_c_ab

In [25]:
L_aa[0][0]/L_ab[0][0]

0.41264340979628666

In [26]:
onsagercalculator.pdbcontainer.iorlist

[(0, array([0.1767767, 0.1767767, 0.       ])),
 (0, array([ 0.1767767, -0.1767767,  0.       ])),
 (0, array([ 0.       , -0.1767767, -0.1767767])),
 (0, array([ 0.       , -0.1767767,  0.1767767])),
 (0, array([0.1767767, 0.       , 0.1767767])),
 (0, array([-0.1767767,  0.       ,  0.1767767]))]

In [ ]:
# we'll look for the (1.0,1.0,0) or x-y oriented dumbbell
xyIdx = None
for idx, (i, o1) in enumerate(onsagercalculator.pdbcontainer.iorlist):
    assert i==0
    if np.allclose(o1, o) or np.allclose(o1, -o):
        idx = i

In [27]:
starInd_g00 = None
tup_g00 = None
starInd_gKin = None
tup_gKin = None

# we'll look for the x-y oriented dumbbell
for starInd, star in enumerate(onsagercalculator.GFstarset_pure):
    for tup in star:
        if np.allclose(tup[1], 0.) and tup[0][0] == tup[0][1] == 0:
            tup_g00 = tup
            starInd_g00 = starInd
            print(starInd_g00, tup_g00)

        elif np.allclose(tup[1], 2.) and tup[0][0] == tup[0][1] == 0:
            tup_gKin = tup
            starInd_gKin = starInd
            print(starInd_gKin, tup_gKin)

1 ((2, 2), array([2., 2., 2.]))
6 ((2, 2), array([0., 0., 0.]))


In [37]:
print('kpt\tNkpt\tG(0)\tG(R)\tG(R)-G(0)')
Pmax_GF_Data = {pmaxerror:[] for pmaxerror in range(-10,-5)}
GF_data = []
for nmax in range(1, 13):
    GFcalc_pure = GFcalc.GF_dumbbells(onsagercalculator.pdbcontainer, onsagercalculator.jnet0_indexed, Nmax=nmax)
    for pmax in sorted(Pmax_GF_Data.keys(), reverse=True):
        GFcalc_pure.SetRates(predb0, bFdb0, preT0, bFT0, 10**(pmax))
        Nreduce, Nkpt, kpt = GFcalc_pure.Nkpt, np.prod(GFcalc_pure.kptgrid), GFcalc_pure.kptgrid
        g0 = GFcalc_pure(tup_g00[0][0], tup_g00[0][1], tup_g00[1])
        gR = GFcalc_pure(tup_gKin[0][0], tup_gKin[0][1], tup_gKin[1])
        Pmax_GF_Data[pmax].append((Nkpt, Nreduce, g0, gR))
    
    Nkpt, Nreduce, g0, gR = Pmax_GF_Data[-8][-1] # get the 1e-8 values
    GF_data.append((Nkpt, Nreduce, g0, gR))
    print("{k[0]}x{k[1]}x{k[2]}\t".format(k=kpt) + 
          " {:5d} ({})\t{:.12f}\t{:.12f}\t{:.12f}".format(Nkpt, Nreduce, 
                                                               g0, gR,g0-gR))

kpt	Nkpt	G(0)	G(R)	G(R)-G(0)
6x6x6	   216 (16)	-0.151412013464	-0.008388832684	-0.143023180780
10x10x10	  1000 (48)	-0.151262509102	-0.002422301213	-0.148840207890
14x14x14	  2744 (109)	-0.151257955867	-0.002386185418	-0.148871770449
18x18x18	  5832 (210)	-0.151257350380	-0.002385037675	-0.148872312705
22x22x22	 10648 (363)	-0.151257202450	-0.002384852600	-0.148872349849
26x26x26	 17576 (580)	-0.151257154493	-0.002384798793	-0.148872355700
30x30x30	 27000 (860)	-0.151257135826	-0.002384778779	-0.148872357048
34x34x34	 39304 (1228)	-0.151257127554	-0.002384770105	-0.148872357449
38x38x38	 54872 (1689)	-0.151257123506	-0.002384765917	-0.148872357589
42x42x42	 74088 (2254)	-0.151257121370	-0.002384763723	-0.148872357647
46x46x46	 97336 (2934)	-0.151257120170	-0.002384762496	-0.148872357674
50x50x50	 125000 (3742)	-0.151257119463	-0.002384761774	-0.148872357689


In [28]:
print('kpt\tNkpt\tG(0)\tG(R)\tG(R)-G(0)')
Pmax_GF_Data = {pmaxerror:[] for pmaxerror in range(-10,-5)}
GF_data = []
for nmax in range(1, 13):
    GFcalc_pure = GFcalc.GF_dumbbells(onsagercalculator.pdbcontainer, onsagercalculator.jnet0_indexed, Nmax=nmax)
    for pmax in sorted(Pmax_GF_Data.keys(), reverse=True):
        GFcalc_pure.SetRates(predb0, bFdb0, preT0, bFT0, 10**(pmax))
        Nreduce, Nkpt, kpt = GFcalc_pure.Nkpt, np.prod(GFcalc_pure.kptgrid), GFcalc_pure.kptgrid
        g0 = GFcalc_pure(tup_g00[0][0], tup_g00[0][1], tup_g00[1])
        gR = GFcalc_pure(tup_gKin[0][0], tup_gKin[0][1], tup_gKin[1])
        Pmax_GF_Data[pmax].append((Nkpt, Nreduce, g0, gR))
    
    Nkpt, Nreduce, g0, gR = Pmax_GF_Data[-8][-1] # get the 1e-8 values
    GF_data.append((Nkpt, Nreduce, g0, gR))
    print("{k[0]}x{k[1]}x{k[2]}\t".format(k=kpt) + 
          " {:5d} ({})\t{:.12f}\t{:.12f}\t{:.12f}".format(Nkpt, Nreduce, 
                                                               g0, gR,g0-gR))

kpt	Nkpt	G(0)	G(R)	G(R)-G(0)
6x6x6	   216 (16)	-0.151412013464	-0.008388832684	-0.143023180780
10x10x10	  1000 (48)	-0.151262509102	-0.002422301213	-0.148840207890
14x14x14	  2744 (109)	-0.151257955867	-0.002386185418	-0.148871770449
18x18x18	  5832 (210)	-0.151257350380	-0.002385037675	-0.148872312705
22x22x22	 10648 (363)	-0.151257202450	-0.002384852600	-0.148872349849
26x26x26	 17576 (580)	-0.151257154493	-0.002384798793	-0.148872355700
30x30x30	 27000 (860)	-0.151257135826	-0.002384778779	-0.148872357048
34x34x34	 39304 (1228)	-0.151257127554	-0.002384770105	-0.148872357449
38x38x38	 54872 (1689)	-0.151257123506	-0.002384765917	-0.148872357589
42x42x42	 74088 (2254)	-0.151257121370	-0.002384763723	-0.148872357647
46x46x46	 97336 (2934)	-0.151257120170	-0.002384762496	-0.148872357674
50x50x50	 125000 (3742)	-0.151257119463	-0.002384761774	-0.148872357689


In [29]:
# save the data so that we don't have it all the time
import pickle
with open("GF_data.pkl", "wb") as fl:
    pickle.dump(GF_data, fl)

with open("Pmax_GF_Data.pkl", "wb") as fl:
    pickle.dump(Pmax_GF_Data, fl)

In [30]:
def fit_g(y, Nkpoints, expons, weight_fac=1):
    y_pred = np.zeros((expons.shape[0], Nkpoints.shape[0]))
    yinf_pred = np.zeros(expons.shape[0])
    losses = np.zeros(expons.shape[0])
    for expInd in range(expons.shape[0]):
        exp = expons[expInd]
        x = Nkpoints**(-exp)
        x2 = x * x
        
        wt = Nkpoints**(weight_fac*exp)
        p = wt / np.sum(wt)
        
        xavg, x2avg = np.sum(p*x), np.sum(p*x2)
        yavg, xyavg = np.sum(p*y), np.sum(p*x*y)
        
        alpha = (xyavg - xavg*yavg)/(x2avg - xavg**2)
        beta = yavg - alpha * xavg
        
        yinf_pred[expInd] = beta
        y_pred[expInd, :] = alpha * x + beta

        losses[expInd] = np.linalg.norm(y - y_pred[expInd, :])**2 / (y.shape[0])
    
    return y_pred, yinf_pred, losses

In [31]:
print('pmax\tG0inf')
for pmax in sorted(Pmax_GF_Data.keys(), reverse=True):
    
    # get the GF data for the pmax value 
    data = Pmax_GF_Data[pmax]
    
    Nkpt = np.array([N for (N,Nr, g0, gR) in data])
    y_0 = np.array([g0 for (N,Nr, g0, gR) in data])
    
    # vary the fit exponents between a range
    # g = (alpha * N^(-exponent) + Ginf)
    exponents = np.arange(1.5, 5, 0.01)
    
    # Get the weighted least squares values for the exponents of g0
    # weight = N^(weight_fac * exponent) for each exponent
    g0_pred, g0_inf, loss_g0 = fit_g(y_0, Nkpt, exponents, weight_fac=1)
    
    # See which exponent gives minimum loss
    # and get the g0inf for that
    mn0 = np.argmin(loss_g0)
    G0inf = g0_inf[mn0]
    
    print('{}\t{}'.format(pmax, G0inf))

pmax	G0inf
-6	-0.15125712409502823
-7	-0.15125712093829882
-8	-0.15125712069635852
-9	-0.15125712076269401
-10	-0.1512571208286163


In [32]:
pmax = -8
for Nkpt, Nreduce, g0, gR in Pmax_GF_Data[pmax]: # get the 1e-8 values
    print(" {:5d} ({})\t{:.12f}\t{:.12f}\t{:.12f}".format(Nkpt, Nreduce, 
                                                               g0, gR,g0-gR))

   216 (16)	-0.151412013464	-0.008388832684	-0.143023180780
  1000 (48)	-0.151262509102	-0.002422301213	-0.148840207890
  2744 (109)	-0.151257955867	-0.002386185418	-0.148871770449
  5832 (210)	-0.151257350380	-0.002385037675	-0.148872312705
 10648 (363)	-0.151257202450	-0.002384852600	-0.148872349849
 17576 (580)	-0.151257154493	-0.002384798793	-0.148872355700
 27000 (860)	-0.151257135826	-0.002384778779	-0.148872357048
 39304 (1228)	-0.151257127554	-0.002384770105	-0.148872357449
 54872 (1689)	-0.151257123506	-0.002384765917	-0.148872357589
 74088 (2254)	-0.151257121370	-0.002384763723	-0.148872357647
 97336 (2934)	-0.151257120170	-0.002384762496	-0.148872357674
 125000 (3742)	-0.151257119463	-0.002384761774	-0.148872357689
